In [ ]:
import torch
from pathlib import Path
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import loss
import pandas as pd
import json
import numpy as np
import stemdiff as sd

dataset = h5py.File("dataset_filtered/train_target.h5", 'r')

In [ ]:

with open("dataset_filtered/dataset_params.json", "r") as f:
    center_sizes = json.load(f)["center_sizes"]

# Calculate constants

In [ ]:
calibration_constants = {
    'au': 0.03377241772151899, 
    'tbf3': 0.031983907407407405, 
    'feo': 0.031019912500000003, 
    'laf3': 0.03087730158730159, 
    'gdf3': 0.03332153703703704, 
    'tio2-a': 0.03191196855385761, 
    'tio2-r': 0.03171350474250266
}

# Verify different scale and save target profiles

In [ ]:
target_profiles = {}
for scale_factor in [1, 2, 4]:
    for n, c in calibration_constants.items():
        df = pd.read_csv(Path("dataset") / n , sep=r'\s+')
        q = torch.from_numpy(df.q.to_numpy())
        I = torch.from_numpy(df.I.to_numpy())
        target1d = loss.resize_target(q, I, (1 / calibration_constants[n]) / (4 / scale_factor))

        # Save target profiles
        if scale_factor == 1:
            target_profiles[n] = target1d
        else:
            target_profiles[f"{n}x{scale_factor}"] = target1d

In [ ]:
scale_factor = 4
for n, d in dataset.items():
    print(n)
    print(len(d))
    input2d = torch.from_numpy(d[:1000])[:, None].float()
    
    db = sd.dbase.read_database(Path("dataset_filtered") / "dbase" / f"db_train_{n}")
    db = db.iloc[:1000]
    centers = db[["Xcenter", "Ycenter"]].to_numpy() / 4

    if scale_factor == 1:
        prof = target_profiles[n]
    else:
        prof = target_profiles[f"{n}x{scale_factor}"]

    target = (prof[None], torch.tensor([9]), centers)
    rad_dist = loss.RadialDistribution(256 * scale_factor, 256 * scale_factor, input2d.device)
    with torch.no_grad():
        device = input2d.device
        intensity, target1d = loss.prepare_profiles(input2d, target, False, rad_dist, scale_factor)
        target1d = target1d.squeeze()
    
    # plt.imshow(torch.log10_(summed_input2d[0, 0] + 1).cpu())
    # plt.show()
    plt.plot(target1d[:100 * scale_factor])
    plt.plot(intensity[:100 * scale_factor])
    plt.show()

    del input2d

## Save profile files

In [ ]:
# Show saved profiles
target_profiles.keys()

In [ ]:
for n, p in target_profiles.items():
    output_dir1 = Path("dataset/target_profiles/")
    output_dir2 = Path("dataset_filtered/target_profiles/")
    output_dir3 = Path("dataset_all/target_profiles/")

    output_dir1.mkdir(parents=True, exist_ok=True)
    output_dir2.mkdir(parents=True, exist_ok=True)
    output_dir3.mkdir(parents=True, exist_ok=True)
    
    np.savetxt(output_dir1 / n, p)
    np.savetxt(output_dir2 / n, p)
    np.savetxt(output_dir3 / n, p)